In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import normalize
from collections import defaultdict

H5_PATH = "../embeddings/kga_dinov2l_embeddings_v2.h5"
N_BLOCKS = 5
K_FACTOR = 0.5
INIT_TRAIN_RATIO = 0.8   # fraction of Block 1 used to initialise prototypes

print("Configuration set.")
print(f"  H5 path        : {H5_PATH}")
print(f"  Blocks         : {N_BLOCKS}")
print(f"  k_factor       : {K_FACTOR}")
print(f"  Init train frac: {INIT_TRAIN_RATIO}")

In [ ]:
import pandas as pd
import numpy as np
import h5py

with h5py.File(H5_PATH, "r") as hf:
    # 1. Load data
    raw_embeddings = hf["embeddings"][:]
    raw_species = np.array([s.decode() for s in hf["species"][:]])
    raw_strings = [t.decode() for t in hf["date_captured"][:]]
    
    # 2. Convert and Create Mask
    # Synchronize dropping NaT values across all arrays
    temp_times = pd.to_datetime(raw_strings, errors='coerce')
    mask = temp_times.notna()
    
    # 3. Apply Mask & Normalize
    embeddings = normalize(raw_embeddings[mask], norm="l2")
    species = raw_species[mask]
    timestamps = temp_times[mask]

print(f"Loaded {len(embeddings):,} embeddings after cleaning.")
print(f"Dropped {len(mask) - mask.sum()} rows with invalid timestamps.")

In [ ]:
print(set(raw_species))

In [ ]:
x = {'reptilesamphibians', 'wildebeestblue', 'foxcape', 'honeybadger', 'bustardkori', 'cheetah', 'ostrich', 'birdofprey', 'jackalblackbacked', 'lionfemale', 'harecape', 'steenbok', 'kudu', 'hyenaspotted', 'caracal', 'secretarybird', 'hartebeestred', 'meerkatsuricate', 'porcupine', 'aardvarkantbear', 'duikercommongrey', 'eland', 'birdother', 'hyenabrown', 'leopard', 'foxbateared', 'catafricanwild', 'rodent', 'gemsbokoryx'}
y = {'dikdik', 'monkeyvervet', 'zebra', 'waterbuck', 'cheetah', 'duiker', 'gazellegrants', 'jackalblackbacked', 'lionfemale', 'hippopotamus', 'warthog', 'hyenaspotted', 'lionmale', 'wildebeest', 'porcupine', 'gazellethomsons', 'bushbuck', 'eland', 'klipspringer', 'birdother', 'civet', 'leopard', 'hartebeest', 'foxbateared', 'mongoose', 'oribi', 'giraffe', 'impala', 'topi', 'elephant', 'aardvark', 'haresavannah', 'buffalo', 'baboon', 'domesticanimal'}

print(x.intersection(y))

In [ ]:
import requests

# The URL of the CSV file
url = "https://lila.science/public/lila-taxonomy-mapping_release.csv"

# The name you want to save the file as locally
filename = "lila-taxonomy-mapping.csv"

try:
    print(f"Downloading {url}...")
    
    # Send a GET request to the URL
    response = requests.get(url)
    
    # Check if the request was successful (Status Code 200)
    response.raise_for_status()
    
    # Write the content to a local file
    with open(filename, "wb") as file:
        file.write(response.content)
        
    print(f"Successfully downloaded and saved as: {filename}")

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

In [ ]:
df = pd.read_csv('json_files/lila-taxonomy-mapping.csv')
df.head()

In [ ]:
df.dataset_name.unique()

In [ ]:
import matplotlib.pyplot as plt

# --- 1. Load Location Data (Add this to your loading block) ---
with h5py.File(H5_PATH, "r") as hf:
    # Assuming "location" is stored as bytes in the H5 file
    raw_locations = np.array([l.decode() for l in hf["location"][:]])
    
# --- 2. Apply the same mask used for your other variables ---
locations = raw_locations[mask]

# --- 3. Organize data for plotting ---
# Create a DataFrame for easier grouping
df = pd.DataFrame({
    'location': locations,
    'timestamp': timestamps
})

# Calculate the start and end times for each location
timeline_df = df.groupby('location')['timestamp'].agg(['min', 'max']).reset_index()
timeline_df = timeline_df.sort_values(by='min') # Sort by start date for a cleaner look

# --- 4. Plotting ---
plt.figure(figsize=(10, 8))

for i, row in timeline_df.iterrows():
    # Plot a line from min to max timestamp for each location
    plt.plot([row['min'], row['max']], [row['location'], row['location']], 
             marker='o', markersize=4, linewidth=2)

plt.title('Data Collection Periods by Location', fontsize=14)
plt.xlabel('Date/Time', fontsize=12)
plt.ylabel('Location', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import h5py
from sklearn.preprocessing import normalize

H5_PATH = "../embeddings/kru_dinov2l_embeddings_v2.h5"  # <-- update this
OUTLIER_LOCATIONS = {"24", "27"}

# --- 1. Load everything ---
with h5py.File(H5_PATH, "r") as hf:
    raw_embeddings = hf["embeddings"][:]
    raw_species    = np.array([s.decode() for s in hf["species"][:]])
    raw_strings    = [t.decode() for t in hf["date_captured"][:]]
    raw_locations  = np.array([l.decode() for l in hf["location"][:]])

# --- 2. Build the same timestamp mask as before ---
temp_times = pd.to_datetime(raw_strings, errors="coerce")
timestamp_mask = temp_times.notna()

# --- 3. Build the outlier-location mask (on the RAW arrays) ---
location_mask = ~np.isin(raw_locations, list(OUTLIER_LOCATIONS))

# --- 4. Combine both masks ---
final_mask = timestamp_mask & location_mask

print(f"Total rows        : {len(raw_embeddings):,}")
print(f"After ts  filter  : {timestamp_mask.sum():,}")
print(f"After loc filter  : {final_mask.sum():,}")
print(f"Rows removed      : {(~final_mask).sum():,}")

# --- 5. Apply mask to every array ---
clean_embeddings = raw_embeddings[final_mask]
clean_species    = raw_species[final_mask]
clean_dates      = np.array(raw_strings)[final_mask]
clean_locations  = raw_locations[final_mask]

# --- 6. Save to a new H5 file (safe — keeps the original intact) ---
OUTPUT_PATH = "../embeddings/kru_dinov2l_embeddings_v2.h5"

with h5py.File(OUTPUT_PATH, "w") as hf:
    hf.create_dataset("embeddings",    data=clean_embeddings)
    hf.create_dataset("species",       data=np.array(clean_species,   dtype="S"))
    hf.create_dataset("date_captured", data=np.array(clean_dates,     dtype="S"))
    hf.create_dataset("location",      data=np.array(clean_locations, dtype="S"))

print(f"\nSaved cleaned file → {OUTPUT_PATH}")

In [1]:
from pathlib import Path, PurePosixPath
import os, json, zipfile, urllib.request
import pandas as pd

def dataset_path_to_local_name(path_like: str) -> str:
    normalized = str(path_like).replace("\\", "/")
    return str(PurePosixPath(normalized)).replace("/", "_")

dataset = 'SnapshotSerengetiS02'

#metadata_url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/snapshotserengeti-v-2-0/SnapshotSerengetiS01.json.zip"
metadata_url = 'https://lilawildlife.blob.core.windows.net/lila-wildlife/snapshotserengeti-v-2-0/SnapshotSerengetiS02.json.zip'
zip_file = "snapshot_serengeti_s2.json.zip"
json_file = f"./json_files/{dataset}.json"
EMPTY_LABELS = {"empty", "human", "blank"}
BEHAVIOR_COLS = ["standing", "resting", "moving", "interacting", "young_present"]

if not os.path.exists(json_file):
    print("Downloading metadata...")
    urllib.request.urlretrieve(metadata_url, zip_file)

    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(os.path.dirname(json_file))
    os.remove(zip_file)

print("Loading Dataset Metadata...")
with open(json_file) as f:
    metadata = json.load(f)

cat_to_name = {c["id"]: c["name"].lower().strip() for c in metadata["categories"]}
image_map = {img["id"]: img for img in metadata["images"]}

valid_images = 0
non_animal_images = 0
non_behavior_images = 0
pos_threshold = 0.5

behavior_counts = {
    behavior: {
        "positive": 0,
        "negative": 0,
        "missing": 0
    }
    for behavior in BEHAVIOR_COLS
}

for ann in metadata["annotations"]:
    species = cat_to_name.get(ann["category_id"], "unknown")
    if (species in EMPTY_LABELS) or (species == "unknown"):
        non_animal_images += 1
        continue

    img = image_map.get(ann["image_id"])
    if not img:
        continue

    has_missing_behavior = False

    for behavior in BEHAVIOR_COLS:
        value = ann.get(behavior)
        if pd.isna(value):
            behavior_counts[behavior]["missing"] += 1
            has_missing_behavior = True
        else:
            value = float(value)
            if value > pos_threshold:
                behavior_counts[behavior]["positive"] += 1
            else:
                behavior_counts[behavior]["negative"] += 1

    if has_missing_behavior:
        non_behavior_images += 1
        continue

    valid_images += 1

print(f"Valid images: {valid_images}")
print(f"Non-animal images: {non_animal_images}")
print(f"Images with missing behavior labels: {non_behavior_images}")

print("\nBehavior counts:")
for behavior, counts in behavior_counts.items():
    total_labeled = counts["positive"] + counts["negative"]
    print(
        f"{behavior:15s} "
        f"pos={counts['positive']:6d}  "
        f"neg={counts['negative']:6d}  "
        f"missing={counts['missing']:6d}  "
        f"labeled={total_labeled:6d}"
    )

Loading Dataset Metadata...
Valid images: 196932
Non-animal images: 386296
Images with missing behavior labels: 0

Behavior counts:
standing        pos= 79179  neg=117753  missing=     0  labeled=196932
resting         pos= 18414  neg=178518  missing=     0  labeled=196932
moving          pos= 83701  neg=113231  missing=     0  labeled=196932
interacting     pos=  1270  neg=195662  missing=     0  labeled=196932
young_present   pos=  3754  neg=193178  missing=     0  labeled=196932
